# Layer 3 — Growth Allocation Matrix

> **Project:** Amazon Revenue Analytics — *A BI Framework for Finance Decision Support*
> **Source:** MIT Media Lab Amazon purchase dataset (Berke / Calacci / Larson)
> **Panel:** 2,846 U.S. Amazon households (2018–2022; cohort-capped at 2023-01-01)
> **Layer 3 question (Q3):** *Which categories deserve priority investment given current growth × scale dynamics?*

**Role boundary:** Surface a category-level allocation matrix that Finance can use as decision-support input. The analyst does NOT own the budget-reallocation decision; the matrix is one of several inputs alongside category-level cost data, supply-chain commitments, and competitive positioning — none of which are in this panel.

**Vocabulary discipline:** No "forecast" / "we recommend" / "investment priority". Findings are surfaced as "the data suggests X is in the Y quadrant" — Finance owns the decision.

**Cross-layer integration:** Layer 3 absorbs Layer 1 (decile structure) and Layer 2 (forward RaR exposure) into category-level allocation. The Layer 1+2 lens is what makes the simple Scale × Growth quadrant interesting — without it, the matrix would just confirm "Pet/Health invest, Books maintain" obvious from any retail growth-share matrix.

## Task 8.1 — Category taxonomy rollup (1,816 → 12 super-categories)

Used Claude Opus 4.7 to build a deterministic taxonomy mapping from the 1,816 raw Amazon browse-node leaf categories to 12 super-categories. Mapping committed at `outputs/tables/category_taxonomy.json` (and flattened CSV at `outputs/tables/category_taxonomy_mapping.csv` for SQL JOIN).

**Coverage (cohort-capped panel 2018-2022, $24.4M GMV):**

| Coverage bucket | GMV % |
|---|---|
| Mapped to specific super-category (11 of 12) | 89.0% |
| NULL bucket (raw `Category` field missing — explicit observable) | 5.4% |
| `Other / Unknown` (specialized long-tail leaves not cleanly mappable) | 5.7% |

**Methodology:**
- Priority-ordered keyword matching (Pet rules before Grocery to resolve PET_FOOD; ABIS_ rule restricted to known media variants to avoid ABIS_DRUGSTORE → Books false positive)
- Explicit category lookup for short-form labels (BRA, HAT, ROBE) where substring rules don't catch
- 50-row random spot-check audit; 3 false positives identified and patched (ABIS_DRUGSTORE → Health, TREE_SKIRT → Home, TEAPOT → Home)
- `__NULL__` mapped to `Other / Unknown` by SQL `COALESCE` — never silently dropped

**Audit reproducibility:** The taxonomy JSON contains both `raw_to_super` and `super_to_raw` dicts plus a `coverage_summary` with frozen percentages. A reviewer can `cat outputs/tables/category_taxonomy.json | jq '.coverage_summary'` and diff against the values above.

In [1]:
import json, os
from pathlib import Path
import polars as pl
import numpy as np

# Notebook runs from notebooks/; chdir to project root for relative paths
PROJECT_ROOT = Path('/Users/leowan34/Documents/amazon-revenue-analytics')
os.chdir(PROJECT_ROOT)

tax = json.loads(Path('outputs/tables/category_taxonomy.json').read_text())
print(f"Taxonomy version: {tax['version']}")
print(f"Generator:        {tax['generator']}")
print(f"n_raw:            {tax['n_raw_categories']:,}")
print(f"n_super:          {tax['n_super_categories']}")
print()
print("Coverage summary:")
for k, v in tax['coverage_summary'].items():
    print(f"  {k}: {v}")

Taxonomy version: 5.0-final
Generator:        Claude Opus 4.7 (Layer 3 Task 8.1, final after spot-check fixes)
n_raw:            1,817
n_super:          12

Coverage summary:
  total_gmv_cohort_capped: 24443100.0
  null_bucket_gmv: 1307881.08
  null_bucket_pct: 5.35
  true_unmapped_gmv: 1391679.3
  true_unmapped_pct: 5.69
  specific_super_pct: 88.96


## Task 8.2 — SQL rollup: super-category × year aggregates

`sql/07_category_rollup.sql` joins cohort-capped purchases against the taxonomy CSV and aggregates GMV / transaction count / unique household count per (super-category, year). NULL raw category is COALESCEd to `Other / Unknown` so panel-GMV reconciles exactly to Layer 1's cohort-capped total ($24,443,100).

In [2]:
import duckdb
con = duckdb.connect()

# Execute Layer 3 rollup SQL
sql = Path('sql/07_category_rollup.sql').read_text()
rollup = con.execute(sql).pl()
print(f"rollup rows: {rollup.height}")

# Reconcile total against Layer 1
total = rollup['total_gmv'].sum()
print(f"Total GMV across all super-cat × year: ${total:,.2f}")
print(f"Layer 1 cohort-capped panel total:     $24,443,100.00")
assert abs(total - 24_443_100) < 1.0, "total drift from Layer 1!"
print("RECONCILIATION: ✓ exact match")
print()

# Pivot to wide year × super-cat view
pivoted = rollup.pivot(on='year', index='super_category', values='total_gmv')
pl.Config.set_tbl_rows(15)
print(pivoted)

rollup rows: 60
Total GMV across all super-cat × year: $24,443,100.00
Layer 1 cohort-capped panel total:     $24,443,100.00
RECONCILIATION: ✓ exact match

shape: (12, 6)
┌────────────────────────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ super_category                 ┆ 2018      ┆ 2019      ┆ 2020      ┆ 2021      ┆ 2022      │
│ ---                            ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str                            ┆ f64       ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════════════════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Apparel & Footwear             ┆ 314608.31 ┆ 423643.48 ┆ 476324.19 ┆ 716993.02 ┆ 763455.59 │
│ Auto, Tools & Outdoor          ┆ 99034.91  ┆ 127924.32 ┆ 200003.78 ┆ 259450.37 ┆ 251255.52 │
│ Books & Media                  ┆ 179907.85 ┆ 165907.03 ┆ 209605.2  ┆ 221705.9  ┆ 191072.44 │
│ Electronics & Accessories      ┆ 559874.39 ┆ 647228.93 ┆ 905391.58 ┆

## Task 8.3 — Scale + Growth + Volatility + Per-household scale (with bootstrap CIs)

Per super-category compute:

| Metric | Definition |
|---|---|
| `scale_2022_gmv` | 2022 GMV — current footprint |
| `cagr_4y` | 4-year CAGR = `(2022_GMV / 2018_GMV)^(1/4) − 1` — secular growth |
| `growth_2020` | `(2022_GMV − 2020_GMV) / 2020_GMV` — short-window growth (alt baseline) |
| `volatility_cv` | Coefficient of variation across 5 annual GMV points — steady vs lumpy |
| `per_hh_scale_2022` | 2022 GMV / N households buying — intensity vs broad-base |
| Bootstrap 95% CI | B=1000 household-resamples with seed=42 |

**Why CAGR vs 2020-baseline growth:** 2020 was already COVID-lifted; using 2020 as denominator underestimates true secular growth. 4-year CAGR uses the full pre-COVID-to-COVID-recovery panel, capturing real trend rather than a post-shock snapshot.

In [3]:
sg = pl.read_parquet('outputs/tables/category_scale_growth.parquet')
pl.Config.set_tbl_rows(15); pl.Config.set_fmt_str_lengths(35)
# Display sorted by 4-year CAGR descending
sg_sorted = sg.sort('cagr_4y', descending=True).select([
    pl.col('super_category'),
    (pl.col('scale_2022_gmv')/1000).round(0).alias('scale_2022_$K'),
    (pl.col('cagr_4y')*100).round(2).alias('CAGR_4y_%'),
    (pl.col('cagr_ci_low')*100).round(2).alias('CAGR_CI_lo'),
    (pl.col('cagr_ci_high')*100).round(2).alias('CAGR_CI_hi'),
    (pl.col('growth_2020')*100).round(1).alias('Growth_20_%'),
    pl.col('volatility_cv').round(3).alias('Vol_CV'),
    pl.col('per_hh_scale_2022').round(0).alias('Per_HH_$'),
])
print(sg_sorted)

shape: (12, 8)
┌─────────────┬─────────────┬───────────┬────────────┬────────────┬────────────┬────────┬──────────┐
│ super_categ ┆ scale_2022_ ┆ CAGR_4y_% ┆ CAGR_CI_lo ┆ CAGR_CI_hi ┆ Growth_20_ ┆ Vol_CV ┆ Per_HH_$ │
│ ory         ┆ $K          ┆ ---       ┆ ---        ┆ ---        ┆ %          ┆ ---    ┆ ---      │
│ ---         ┆ ---         ┆ f64       ┆ f64        ┆ f64        ┆ ---        ┆ f64    ┆ f64      │
│ str         ┆ f64         ┆           ┆            ┆            ┆ f64        ┆        ┆          │
╞═════════════╪═════════════╪═══════════╪════════════╪════════════╪════════════╪════════╪══════════╡
│ Grocery &   ┆ 507.0       ┆ 34.31     ┆ 30.66      ┆ 37.95      ┆ 32.6       ┆ 0.417  ┆ 252.0    │
│ Food &      ┆             ┆           ┆            ┆            ┆            ┆        ┆          │
│ Beverage    ┆             ┆           ┆            ┆            ┆            ┆        ┆          │
│ Pet         ┆ 291.0       ┆ 27.25     ┆ 23.96      ┆ 31.1       ┆ 23.5    

## Task 8.4 — Cross-layer crosswalk (Layer 1 deciles + Layer 2 RaR)

Per super-category compute:

| Metric | Definition |
|---|---|
| `d1_gmv_share` | % of category GMV from D1 (top decile) households — VIP concentration |
| `d10_gmv_share` | % from D10 (bottom decile) households — long-tail bottom |
| `mid_decile_gmv_share` | % from D6-D9 households — Layer 2's RaR-heavy mid-segment |
| `mid_decile_rar_share` | % of category total RaR from D6-D9 households — forward risk exposure |
| `household_breadth` | n_households_buying / 2,845 — horizontal participation |

**This is the Layer 3 differentiator.** A naive Scale × Growth matrix would put Pet Food in INVEST and Books in MAINTAIN. The Layer 1+2 crosswalk reframes that: Pet is D1-concentrated (VIP retention) while Books has highest mid-decile share (broad-base retention).

In [4]:
cross = pl.read_parquet('outputs/tables/category_layer_crosswalk.parquet')
print(cross.sort('d1_gmv_share', descending=True).select([
    pl.col('super_category'),
    (pl.col('d1_gmv_share')*100).round(1).alias('D1_%'),
    (pl.col('d10_gmv_share')*100).round(1).alias('D10_%'),
    (pl.col('mid_decile_gmv_share')*100).round(1).alias('Mid_GMV_%'),
    (pl.col('mid_decile_rar_share')*100).round(1).alias('Mid_RaR_%'),
    (pl.col('household_breadth')*100).round(1).alias('Breadth_%'),
    pl.col('total_category_rar').round(0).alias('Cat_RaR_$'),
]))

shape: (12, 7)
┌────────────────────────────────┬──────┬───────┬───────────┬───────────┬───────────┬───────────┐
│ super_category                 ┆ D1_% ┆ D10_% ┆ Mid_GMV_% ┆ Mid_RaR_% ┆ Breadth_% ┆ Cat_RaR_$ │
│ ---                            ┆ ---  ┆ ---   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str                            ┆ f64  ┆ f64   ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞════════════════════════════════╪══════╪═══════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Auto, Tools & Outdoor          ┆ 40.5 ┆ 0.4   ┆ 10.8      ┆ 63.4      ┆ 83.9      ┆ 22138.0   │
│ Pet                            ┆ 39.9 ┆ 0.2   ┆ 9.6       ┆ 59.7      ┆ 66.7      ┆ 15540.0   │
│ Home, Kitchen & Bath           ┆ 39.1 ┆ 0.4   ┆ 11.2      ┆ 65.3      ┆ 96.0      ┆ 27986.0   │
│ Office, Stationery & Crafts    ┆ 38.8 ┆ 0.6   ┆ 12.5      ┆ 64.4      ┆ 85.0      ┆ 22548.0   │
│ Toys, Games & Hobbies          ┆ 38.1 ┆ 0.6   ┆ 13.1      ┆ 64.5      ┆ 88.8      ┆ 24002.0   │
│ Gro

## Task 8.5 — Cohort gateway analysis (acquisition vs loyalty)

**Cohort definition:**
- **New cohort:** First transaction date on or after 2020-01-01 (post-COVID joiners, n=196, 6.9% of panel)
- **Established cohort:** First transaction before 2020-01-01 (n=2,649, 93.1% of panel)

**Gateway lift per category:**
```
lift = new_cohort_penetration / established_cohort_penetration
```

**Reading the lift:** All lifts are < 1.0 in this panel — every category sees lower penetration in the new cohort (196 households) than the established cohort (2,649 households) because new cohort has had less time to expand category exploration. The **ranking** is what matters:

- **Highest lift = acquisition gateway** (new cohort adopts rapidly — fastest first-purchase surface)
- **Lowest lift = loyalty category** (slow new-cohort adoption — established-customer-anchored)

Bootstrap 95% CI on lift via B=1000 household-resamples, seed=42.

In [5]:
gateway = pl.read_parquet('outputs/tables/category_cohort_gateway.parquet')
print(gateway.sort('gateway_lift', descending=True).select([
    pl.col('super_category'),
    pl.col('new_cohort_buyers').alias('new_buyers'),
    pl.col('established_cohort_buyers').alias('est_buyers'),
    (pl.col('new_penetration')*100).round(1).alias('new_%'),
    (pl.col('established_penetration')*100).round(1).alias('est_%'),
    pl.col('gateway_lift').round(3).alias('lift'),
    pl.col('lift_ci_low').round(3).alias('CI_lo'),
    pl.col('lift_ci_high').round(3).alias('CI_hi'),
]))

shape: (12, 8)
┌────────────────────────────────┬────────────┬────────────┬───────┬───────┬───────┬───────┬───────┐
│ super_category                 ┆ new_buyers ┆ est_buyers ┆ new_% ┆ est_% ┆ lift  ┆ CI_lo ┆ CI_hi │
│ ---                            ┆ ---        ┆ ---        ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   │
│ str                            ┆ i64        ┆ i64        ┆ f64   ┆ f64   ┆ f64   ┆ f64   ┆ f64   │
╞════════════════════════════════╪════════════╪════════════╪═══════╪═══════╪═══════╪═══════╪═══════╡
│ Electronics & Accessories      ┆ 167        ┆ 2609       ┆ 85.2  ┆ 98.5  ┆ 0.865 ┆ 0.812 ┆ 0.913 │
│ Apparel & Footwear             ┆ 159        ┆ 2540       ┆ 81.1  ┆ 95.9  ┆ 0.846 ┆ 0.785 ┆ 0.902 │
│ Home, Kitchen & Bath           ┆ 160        ┆ 2570       ┆ 81.6  ┆ 97.0  ┆ 0.841 ┆ 0.785 ┆ 0.899 │
│ Health, Beauty & Personal Care ┆ 158        ┆ 2560       ┆ 80.6  ┆ 96.6  ┆ 0.834 ┆ 0.779 ┆ 0.886 │
│ Other / Unknown                ┆ 157        ┆ 2611       ┆ 80.1  ┆ 98.6  ┆

## Tasks 8.6 / 8.7 / 8.8 — Three Layer 3 figures

| File | Role | Encoding |
|---|---|---|
| `outputs/figures/category_allocation_matrix.png` | **Layer 3 hero** | x=Scale (log $K), y=CAGR, bubble size=per-household $, color=D1 share |
| `outputs/figures/category_ranking_table.png` | Supporting | Ranked horizontal bar of CAGR with bootstrap CI whiskers |
| `outputs/figures/category_gateway_lift.png` | Supporting | New / established cohort penetration lift |

All 3 generated at 300 DPI; figures regenerate from `outputs/tables/category_*.parquet` deterministically (no LLM call in the figure-build step).

## Task 8.9 — Three surprising findings (cross-layer, locked)

**Strict scope:** BA insights about category-level allocation through the Layer 1+2 lens. NOT audit/engineering wins (those belong in README Methodology Notes).

### Finding 1 — *(VIP-retention reframe)*  Pet is a D1-anchored loyalty category, not a "growth + risk mitigation" investment

A naive Scale × Growth lens puts Pet in the upper-right INVEST quadrant: 4-year CAGR 27% (2nd highest after Grocery), 2022 scale $291K, per-household $216. But the Layer 1+2 crosswalk reframes Pet as the most **VIP-anchored** super-category in the panel:

- **D1 GMV share = 39.9%** — highest of any super-category (top decile drives 40% of Pet GMV)
- **Mid-decile (D6-D9) GMV share = 9.6%** — lowest of any super-category (mid-tier customers underweight Pet)
- **Mid-decile RaR share = 59.7%** — lowest of any super-category (Pet customers have low forward drop-off exposure)
- **Acquisition gateway lift = 0.51** — lowest of any super-category (new cohort penetrates Pet at half the rate of established cohort)
- **Household breadth = 66.7%** — second-narrowest after Gift Cards

> The data suggests Pet is squarely a **VIP retention** category — high-value, top-decile-anchored, low-churn, loyalty-driven. The "INVEST" recommendation from Scale × Growth alone is operationally about defending existing top-decile customers, not acquiring new ones or mitigating Layer 2's mid-decile RaR concentration.

### Finding 2 — *(harvest-direction reframe)*  Books & Media is declining but anchors broad-base retention — harvesting would worsen mid-decile RaR

Books & Media is the clearest MAINTAIN quadrant entry under the simple lens: 4-year CAGR 1.5% (essentially flat), 2022 scale $191K. The original interview-takeaway framing — *"Books are scale leader but declining — margin maintenance, not growth bet"* — anchors here. But the Layer 1+2 crosswalk surfaces a complication:

- **D1 GMV share = 26.8%** — lowest of any super-category (Books is the **most broad-base** category, NOT top-decile-driven)
- **Mid-decile GMV share = 18.9%** — highest of any super-category
- **Household breadth = 87.7%** — third-highest (after Electronics 98%, Home 96%)
- **Mid-decile RaR share = 63.4%** — at panel average

> The data suggests Books is the closest super-category to a **broad-base retention anchor** in this panel. Harvesting Books per the simple Scale × Growth framework would disproportionately hit the mid-decile customers Layer 2 already identified as panel-RaR-heavy (65% of $29K total RaR). The Layer 3 lens turns a "harvest Books" recommendation into "do not harvest Books — it is mid-decile retention infrastructure".

### Finding 3 — *(acquisition-surface reframe)*  The acquisition gateway is broad-utility, not specialty: Electronics / Apparel / Home / H&PC — Pet/Gift Cards/Auto/Books are loyalty surfaces

The original framing assumed Pet Food + Nutritional Supplements were the clearest INVEST quadrant — high-growth specialty verticals. But the cohort-gateway analysis says new customers (post-2020 joiners, n=196) enter Amazon via broad-utility commodity categories, not specialty verticals:

**Acquisition gateway tier (lift ≥ 0.83 — rapid new-cohort adoption):**
- Electronics & Accessories: lift = 0.865 [CI: 0.812, 0.913]
- Apparel & Footwear:        lift = 0.846 [CI: 0.785, 0.902]
- Home, Kitchen & Bath:      lift = 0.841 [CI: 0.785, 0.899]
- Health, Beauty & PC:       lift = 0.834 [CI: 0.779, 0.886]

**Loyalty tier (lift ≤ 0.65 — slow new-cohort adoption):**
- Pet:                       lift = 0.510 [CI: 0.414, 0.614]
- Gift Cards & Digital:      lift = 0.584 [CI: 0.491, 0.675]
- Auto, Tools & Outdoor:     lift = 0.610 [CI: 0.532, 0.693]
- Books & Media:             lift = 0.635 [CI: 0.558, 0.710]

> The data suggests Amazon's **acquisition surface** is broad-utility daily-need categories (electronics, apparel, home, personal care) — not the specialty verticals (Pet, Books) that the Scale × Growth alone framing would highlight. For Finance: customer-acquisition budget should follow the broad-utility surface; specialty-vertical budget is retention investment.

**Unified Layer 3 narrative for finance leadership:**

> *Twelve super-categories rolled up from 1,816 raw Amazon labels (Claude Opus 4.7 taxonomy, 89% specific-mapped, audit JSON committed). Naive Scale × Growth says "Pet/Health invest, Books maintain". Layer 1+2 reframe: **(1) Pet is VIP retention (D1=40% of Pet GMV, lowest mid-decile share at 9.6%, lowest gateway lift 0.51) — not RaR mitigation**; **(2) Books is broad-base retention anchor (D1=27%, lowest of any super-cat) — harvesting would worsen Layer 2's mid-decile RaR concentration**; **(3) Acquisition surface is broad-utility commodity categories (Electronics/Apparel/Home/H&PC, lift 0.83-0.87) not specialty verticals**. The matrix surfaces decision-support inputs for Finance; the analyst does not own the budget allocation.*

## Task 8.10 — Final synthesis (cross-layer-3 takeaways)

Three layers, one panel, walk-forward methodology, anti-vocab discipline maintained throughout.

**Layer 1 — concentration at the top.** Top decile drives 36.2% of GMV; gap is ~94% purchase frequency, only ~6% basket size. Heavy-cadence over-index +387%.

**Layer 2 — risk is in the middle.** Top decile drives only 0.5% of forward-looking RaR; mid-deciles 6-9 carry 65% of RaR on 13% of GMV (5× amplification; 21× top-vs-bottom asymmetry).

**Layer 3 — allocation reframe under the cross-layer lens.** Naive Scale × Growth quadrant labels (Pet INVEST, Books MAINTAIN) are operationally misleading once Layer 1 + Layer 2 are folded in: Pet is VIP retention, Books is broad-base retention infrastructure, acquisition gateway is broad-utility commodity categories. The data suggests Finance segment budget into three lanes — top-decile retention (Pet, specialty verticals), mid-decile retention/RaR mitigation (Books, Electronics, Home), customer acquisition (commodity broad-utility) — rather than a single "growth bet" allocation per category.

**Open questions for downstream work (NOT in this panel):**

- Sub-category granularity within each super-category (e.g., Health → Supplements vs Skincare vs Medication — Finance may want to bet within H&PC, not on H&PC as a single bucket)
- Multi-quarter outcome window (Layer 2 limitation — single-quarter Q3 drop-off only)
- Production cohort validation (this is a consenting Prolific panel, not Amazon's full customer base)
- Cost-side and supply-chain data (the matrix is a revenue-side input only)

**What this project does NOT claim:** not a forecast (Finance's workflow); not a budget recommendation (Finance owns); not Amazon's full customer base (consenting panel, selection-bias plausible); not multi-quarter churn (single-quarter outcome only); not a category-level revenue-prediction model (rollup framework + crosswalk + bootstrap CIs are surfaced as decision-support inputs).